# **Collaborative Filtering based Recommender System using K Nearest Neighbor**


Collaborative filtering is probably the most commonly used recommendation algorithm. There are two main types of methods: 
 - **User-based** collaborative filtering is based on user similarity or neighborhood
 - **Item-based** collaborative filtering is based on similarity among items


They both work similarly. Let me briefly explain how user-based collaborative filtering works.


User-based collaborative filtering looks for users who are similar. This is very similar to the user clustering method I implemented previously, where I employed explicit user profiles to calculate user similarity. However, user profiles may not always be available, so how can we determine if two users are similar?


#### User-item interaction matrix 


For most collaborative filtering-based recommender systems, the main dataset format is a 2-D matrix called the user-item interaction matrix. In this matrix, rows are labeled with user IDs/indices and columns are labeled with item IDs/indices. The element `(i, j)` represents the rating of user `i` for item `j`.

#### KNN-based collaborative filtering


Each row vector represents the rating history of a user, and each column vector represents the users who rated the item. A user-item interaction matrix is usually very sparse. One user is likely to interact with only a very small subset of all available items, and one item is likely to be interacted with by only a small subset of all users.


To determine if two users are similar, I can simply calculate the similarities between their row vectors in the interaction matrix. Then, based on these similarity measurements, I can find the `k` nearest neighbors as the similar users.


Item-based collaborative filtering works similarly; I just need to look at the user-item matrix vertically. Instead of finding similar users, I'm trying to find similar items (courses). If two courses are enrolled in by two groups of similar users, then I could consider the two items to be similar and use the known ratings from other users to predict the unknown ratings.


The predicted rating of user $u$ for item $i$, $\hat{r}_{ui}$ is given by:


**User-based** collaborative filtering:


$$\hat{r}_{ui} = \frac{
\sum\limits_{v \in N^k_i(u)} \text{similarity}(u, v) \cdot r_{vi}}
{\sum\limits_{v \in N^k_i(u)} \text{similarity}(u, v)}$$


**Item-based** collaborative filtering:


$$\hat{r}_{ui} = \frac{
\sum\limits_{j \in N^k_u(i)} \text{similarity}(i, j) \cdot r_{uj}}
{\sum\limits_{j \in N^k_u(i)} \text{similarity}(i, j)}$$


Here $N^k_i(u)$ denotes the k-nearest neighbors of $u$.


Let me illustrate how the equation works using a simple example. Suppose I want to predict the rating of `user6` for the course `Machine Learning Capstone`. After some similarity measurements, I find that the k = 4 nearest neighbors are: `user2, user3, user4, user5` with similarities in array ```knn_sims```:


In [4]:
import numpy as np
import math

In [5]:
# An example similarity array stores the similarity of user2, user3, user4, and user5 to user6
knn_sims = np.array([0.8, 0.92, 0.75, 0.83])

Also, their ratings on the `Machine Learning Capstone` course are:


In [6]:
# 2.0 means audit and 3.0 means complete the course
knn_ratings = np.array([3.0, 3.0, 2.0, 3.0]) 

So the predicted rating of `user6` for the `Machine Learning Capstone` course can be calculated as:


In [7]:
r_u6_ml = np.dot(knn_sims, knn_ratings) / sum(knn_sims)
r_u6_ml

2.7727272727272725

If I already know the true rating to be 3.0, then I get a prediction error RMSE (Root Mean Squared Error) of:


In [8]:
true_rating = 3.0
rmse = math.sqrt((true_rating - r_u6_ml) ** 2)
rmse

0.22727272727272751

The predicted rating is around 2.7 (close to 3.0 with RMSE 0.28), which indicates that `user6` is also likely to complete the course `Machine Learning Capstone`. As such, I may recommend it to user6 with high confidence.


## Objectives


In this notebook, I will:


* Perform KNN-based collaborative filtering on the user-item interaction matrix


----


### Loading and exploring the dataset


First, let's load our dataset, which is a user-item (learner-course) interaction matrix:


In [9]:
import pandas as pd

In [10]:
rating_df = pd.read_csv("data/ratings.csv")

In [11]:
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3
2,1990814,ML0120ENv3,5
3,380098,BD0211EN,5
4,779563,DS0101EN,3


The dataset contains three columns: `user id` (learner), `item id` (course), and `rating` (enrollment mode).

Note that this matrix is presented in dense or vertical form, and I can convert it to a sparse matrix using `pivot`:


In [12]:
rating_sparse_df = rating_df.pivot(index='user', columns='item', values='rating').fillna(0).reset_index().rename_axis(index=None, columns=None)
rating_sparse_df.head()

,user,AI0111EN,BC0101EN,BC0201EN,BC0202EN,BD0101EN,BD0111EN,BD0115EN,BD0121EN,BD0123EN,...,SW0201EN,TA0105,TA0105EN,TA0106EN,TMP0101EN,TMP0105EN,TMP0106,TMP107,WA0101EN,WA0103EN
0,2,0.0,4.0,0.0,0.0,5.0,4.0,0.0,5.0,3.0,...,0.0,5.0,0.0,4.0,0.0,3.0,3.0,0.0,5.0,0.0
1,4,0.0,0.0,0.0,0.0,5.0,3.0,4.0,5.0,3.0,...,0.0,4.0,0.0,0.0,0.0,3.0,3.0,0.0,3.0,3.0
2,5,3.0,5.0,5.0,0.0,4.0,0.0,0.0,0.0,3.0,...,0.0,0.0,4.0,4.0,4.0,4.0,4.0,5.0,0.0,3.0
3,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Usually, the dense format is preferred as it saves a lot of storage and memory space. The benefit of the sparse matrix is that it's in a natural matrix format, and I can apply computations such as cosine similarity directly.


Next, I need to perform KNN-based collaborative filtering on the user-item interaction matrix. I'll do this using `scikit-surprise`, a popular and easy-to-use Python recommendation system library.


## Using the **Surprise** library


*Surprise* is a Python sci-kit library for recommender systems. It's simple and comprehensive for building and testing different recommendation algorithms.

In [13]:
from surprise import KNNBasic
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

Let's take a look at an example that shows how easily we can perform KNN collaborative filtering on a sample movie review dataset, which contains about 100k movie ratings from users:


In [14]:
# Load the movielens-100k dataset
data = Dataset.load_builtin('ml-100k', prompt=False)

# Sample random trainset and testset
# Test set is made of 25% of the ratings
trainset, testset = train_test_split(data, test_size=.25)

# We'll use the famous KNNBasic algorithm
algo = KNNBasic()

# Train the algorithm on the trainset, and predict ratings for the testset
algo.fit(trainset)
predictions = algo.test(testset)

# Then compute RMSE
accuracy.rmse(predictions)

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /Users/anthony/.surprise_data/ml-100k
Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.9769


0.9769368444908448

As you can see, with just a couple of lines of code, I can apply KNN collaborative filtering on the sample movie lens dataset. The main evaluation metric is `Root Mean Square Error (RMSE)`, which is a very popular rating estimation error metric used in recommender systems as well as many regression model evaluations.


Now, let's load our course rating dataset:


In [15]:
# Save the rating dataframe to a CSV file
rating_df.to_csv("course_ratings.csv", index=False)

# Read the course rating dataset with columns user item rating
reader = Reader(
    line_format='user item rating', sep=',', skip_lines=1, rating_scale=(2, 3))

# Load the dataset from the CSV file
course_dataset = Dataset.load_from_file("course_ratings.csv", reader=reader)

Let's split it into a training set and a test set:


In [16]:
trainset, testset = train_test_split(course_dataset, test_size=.3)

Now let's check how many users and items we can use to fit a KNN model:


In [17]:
print(f"Total {trainset.n_users} users and {trainset.n_items} items in the training set")

Total 31303 users and 124 items in the training set


## KNN-based Collaborative Filtering on the User-Item Interaction Matrix


Now I'll fit the KNN-based collaborative filtering model using the training set and evaluate the results using the test set:


In [23]:
# Define similarity options - using Pearson correlation and item-based approach
sim_options = {"name": "cosine", "user_based": True}

# Create and fit the KNN model
knn_model = KNNBasic(sim_options=sim_options).fit(trainset)

# Make predictions on the test set
predictions = knn_model.test(testset)

# Calculate and display the RMSE
accuracy.rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.2877


1.2877070735894367

## Summary


In this notebook, I've implemented KNN-based collaborative filtering. Though simple, it is still an effective and intuitive collaborative filtering algorithm. Since it's based on KNN, it inherits the main characteristics of KNN, such as being memory-intensive because it needs to maintain a huge similarity matrix among users or items.